# GPR

This notebook trains a Gaussian process regressor to predict $-\Delta G_{abs}$. The prior is taken to be a constant. Results are saved in the `models` folder. SHAP values are also computed. There is an option to use full feature set or a reduced feature set. The reduced feature set is the max_[W] along with totals of each type of bead. The model in the publicaiton has the full dataset with ARD (a length scale for every dimension).

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error as mse
import GPy
import shap
import pickle
import functools
from IPython.display import display
from IPython.display import Image
from matplotlib.pyplot import figure

 /Users/dja3/envir/ii/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning:IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


## Choose model type

In [2]:
ard = True  # True means use a length scale for every dimension
reduced = False  # True means means use only important ones

prefix = ""
if ard:
    prefix += "ard_"
if reduced:
    prefix += "red_"

## Get Data

In [3]:
features = [
    "num_[W]",
    "max_[W]",
    "num_[Tr]",
    "max_[Tr]",
    "num_[Ta]",
    "max_[Ta]",
    "num_[R]",
    "max_[R]",
    "[W]",
    "[Tr]",
    "[Ta]",
    "[R]",
    "rel_shannon",
    "length",
]

In [4]:
df = pd.read_csv("data/disp_dataset.csv")
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values * (-1)

In [5]:
if reduced:

    mask = np.zeros(len(df.columns) - 1).astype(bool)
    mask[8:12] = True

    X_adjust = np.multiply(X[:, mask], X[:, 13][:, np.newaxis])
    X_adjust = np.concatenate([X[:, 1].reshape(-1, 1), X_adjust], axis=1)

else:

    X_adjust = X

print(X_adjust)

[[ 0.          0.          0.4        ...  0.3         0.35616074
  40.        ]
 [ 0.          0.          0.375      ...  0.31578947  0.36186147
  38.        ]
 [ 0.2         2.          0.2        ...  0.35294118  0.36667336
  34.        ]
 ...
 [ 0.75        3.          0.         ...  0.16666667  0.39094785
  24.        ]
 [ 0.8         3.          0.2        ...  0.14285714  0.38324006
  28.        ]
 [ 1.          2.          0.         ...  0.16666667  0.39094785
  24.        ]]


## Train the models

In [6]:
# import warnings
# warnings.simplefilter('error', RuntimeWarning)

In [7]:
for i in range(5):

    # Get the data
    train_mask = np.genfromtxt(
        "data/data_split.csv", delimiter=",", skip_header=1, dtype=bool, usecols=i
    )
    scaler = StandardScaler()
    X_train = X_adjust[train_mask, :]
    X_test = X_adjust[np.logical_not(train_mask), :]
    y_train = y[train_mask]
    y_test = y[np.logical_not(train_mask)]

    X_train_scal = scaler.fit_transform(X_train)
    X_test_scal = scaler.transform(X_test)

    # Build the GP model
    if reduced:
        input_dim = 5
    else:
        input_dim = 14

    kernel = GPy.kern.Matern52(input_dim=input_dim, ARD=ard)
    m = GPy.models.GPRegression(
        X_train_scal, y_train.reshape(-1, 1), kernel, normalizer=True
    )
    m[".*Gaussian_noise_*"] = 0.1

    # Train the model
    m.optimize_restarts(num_restarts=10)
    display(m)

    # Save the model
    with open("models/" + prefix + "fold" + str(i) + ".pkl", "wb") as file:
        pickle.dump(m, file)

 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:137: RuntimeWarning:overflow encountered in square
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:138: RuntimeWarning:invalid value encountered in add
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:586: RuntimeWarning:invalid value encountered in multiply
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:589: RuntimeWarning:invalid value encountered in subtract
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:166: RuntimeWarning:overflow encountered in divide


Optimization restart 1/10, f = -828.5933952918451
Optimization restart 2/10, f = -826.4329333153396
Optimization restart 3/10, f = -826.9631816352385
Optimization restart 4/10, f = -825.2457375358831


 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:243: RuntimeWarning:invalid value encountered in divide


Optimization restart 5/10, f = -827.1515011639144
Optimization restart 6/10, f = -797.5015822407158
Optimization restart 7/10, f = -828.258638911992
Optimization restart 8/10, f = -802.3463181183806
Optimization restart 9/10, f = -827.1350757746661
Optimization restart 10/10, f = -826.5241102650523


GP_regression.,value,constraints,priors
Mat52.variance,100.67187297505176,+ve,
Mat52.lengthscale,"(14,)",+ve,
Gaussian_noise.variance,0.028395265557970083,+ve,


...

 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:166: RuntimeWarning:overflow encountered in divide
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:137: RuntimeWarning:overflow encountered in square
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:138: RuntimeWarning:invalid value encountered in add
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:586: RuntimeWarning:invalid value encountered in multiply
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:589: RuntimeWarning:invalid value encountered in subtract


Optimization restart 1/10, f = -762.3975905934358
Optimization restart 2/10, f = -762.5790361926506
Optimization restart 3/10, f = -764.0218658951298
Optimization restart 4/10, f = -762.8093578305616


 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:243: RuntimeWarning:invalid value encountered in divide


Optimization restart 5/10, f = -758.3295184508061
Optimization restart 6/10, f = -762.0961031003801
Optimization restart 7/10, f = -751.4078264489001
Optimization restart 8/10, f = -763.1831371858943
Optimization restart 9/10, f = -764.0095989532147
Optimization restart 10/10, f = -747.166918116194


GP_regression.,value,constraints,priors
Mat52.variance,96.1370255996028,+ve,
Mat52.lengthscale,"(14,)",+ve,
Gaussian_noise.variance,0.029783489483964167,+ve,


## Look at the results

In [19]:
mse_df = pd.DataFrame(index=["train", "test"], columns=range(5))

for i in range(5):

    # Load model and test/train split
    with open("models/" + prefix + "fold" + str(i) + ".pkl", "rb") as file:
        m = pickle.load(file)
    train_mask = np.genfromtxt(
        "data/data_split.csv", delimiter=",", skip_header=1, dtype=bool, usecols=i
    )

    print(m.parameter_names())
    print(m.param_array)
    print(m.kern.lengthscale)

    # Gather results
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_adjust[train_mask, :])
    X_test = scaler.transform(X_adjust[np.logical_not(train_mask), :])
    yp_train, _ = m.predict(X_train)
    yp_test, _ = m.predict(X_test)
    y_train = y[train_mask]
    y_test = y[np.logical_not(train_mask)]

    mse(y_test, yp_test)

    mse_df.iloc[0, i] = mse(y_train, yp_train)
    mse_df.iloc[1, i] = mse(y_test, yp_test)

['Mat52.variance', 'Mat52.lengthscale', 'Gaussian_noise.variance']
[1.00671873e+02 4.97745428e+02 4.52835569e+01 4.11122661e+02
 7.66802466e+02 6.74793232e+02 7.46558998e+02 7.24023693e+02
 6.31850210e+02 2.26182632e+01 3.39184918e+01 4.76681227e+02
 8.34977561e+01 6.38164736e+02 2.17112491e+01 2.83952656e-02]
  index  |  GP_regression.Mat52.lengthscale  |  constraints  |  priors
  [0]    |                     497.74542775  |      +ve      |        
  [1]    |                      45.28355687  |      +ve      |        
  [2]    |                     411.12266146  |      +ve      |        
  [3]    |                     766.80246614  |      +ve      |        
  [4]    |                     674.79323191  |      +ve      |        
  [5]    |                     746.55899801  |      +ve      |        
  [6]    |                     724.02369331  |      +ve      |        
  [7]    |                     631.85020986  |      +ve      |        
  [8]    |                      22.61826321  |   

In [20]:
mse_df.to_csv("models/" + prefix + "mse.csv")
mse_df.head()

,0,1,2,3,4
train,0.304829,0.312098,0.306968,0.313868,0.315921
test,0.360194,0.322262,0.326009,0.305113,0.312264


## Compute SHAP

In [11]:
def m_mu(X, model):
    mu, var = model.predict(X)
    return mu.flatten()

In [21]:
for i in range(1):

    # Load model and test/train split
    with open("models/" + prefix + "fold" + str(i) + ".pkl", "rb") as file:
        m = pickle.load(file)
    train_mask = np.genfromtxt(
        "data/data_split.csv", delimiter=",", skip_header=1, dtype=bool, usecols=i
    )

    # Find X_test
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_mask, :])
    X_test = scaler.transform(X[np.logical_not(train_mask), :])

    # Compute SHAP values
    explainer = shap.KernelExplainer(
        functools.partial(m_mu, model=m),
        shap.kmeans(X_train, 40),
        feature_names=features,
    )
    shap_values = explainer(X_test)

    # Save SHAP results
    with open("models/shap_values_" + prefix + str(i) + ".pkl", "wb") as f:
        pickle.dump(shap_values, f)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 625/625 [49:14<00:00,  4.73s/it]


In [ ]:
# print(shap_values)